In [ ]:
# test from gemini
# from google.colab import userdata # type: ignore
# import os

# # This fetches the secret you just saved in the Colab UI
# try:
#     os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
#     print("✅ Hugging Face token loaded successfully.")
# except Exception as e:
#     print("❌ Could not find HF_TOKEN in Colab secrets. Check the 'Key' icon in the Colab web UI.")

import os
import getpass

# Run this cell; a text box will appear at the top of VS Code
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF_TOKEN here:")

print("✅ Keys set for this remote session.")

✅ Keys set for this remote session.


In [3]:
from huggingface_hub import notebook_login, whoami
try:
    print(f"Authenticated as: {whoami()['name']}")
except Exception:
    print("❌ Authentication failed. Check your token.")

Authenticated as: kskaneko


In [4]:
# 1. Mount Google Drive
from google.colab import drive # type: ignore
import os
drive.mount('/content/drive')

# 2. Navigate to your project folder (Adjust path if necessary)
%cd /content/drive/MyDrive/Coursework/Spring2026/INFO290/genai-final-project/VisionCart/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Coursework/Spring2026/INFO290/genai-final-project/VisionCart


In [5]:
# 3. Install GPU-specific dependencies
# Note: We use qwen-vl-utils for optimized image processing
%pip install -q transformers accelerate qwen-vl-utils torchvision

In [6]:
import json
import sys
from datetime import datetime

# Add 'src' to path so we can import your agent
sys.path.append(os.path.abspath("src"))
from agents import stylist

def main():
    # 1. Find the latest crawled images
    # Assuming your teammate's crawler saves to data/YYYY-MM-DD...
    data_dir = "data"
    subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    if not subdirs:
        print("No crawled data found in data/ folder!")
        return
    
    latest_session = max(subdirs, key=os.path.getmtime)
    print(f"📂 Analyzing latest session: {latest_session}")

    # Collect all images from the latest crawl
    image_paths = []
    for root, dirs, files in os.walk(latest_session):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(root, file))

    # 2. Prepare the State
    state = {
        "vision_board_paths": image_paths,
        "num_products": 10
    }

    # 3. Execute Stylist Agent (Running on A100/L4 GPU)
    print("🎨 Stylist is analyzing the vision board...")
    result = stylist.run(state)
    
    # 4. Save Output for the Procurement Agent
    output_path = "data/stylist_output.json"
    with open(output_path, "w") as f:
        json.dump(result["stylist_output"], f, indent=2)
    
    print(f"✅ Success! Style profile saved to {output_path}")
    print(json.dumps(result["stylist_output"], indent=2))

if __name__ == "__main__":
    main()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


📂 Analyzing latest session: data/test_samples
🎨 Stylist is analyzing the vision board...
JSON Parsing Error: Extra data: line 10 column 1 (char 414)
✅ Success! Style profile saved to data/stylist_output.json
{
  "style_profile": "Fallback: Athletic/Sporty",
  "products": [
    "jerseys",
    "cleats"
  ],
  "aesthetic": [
    "sporty"
  ],
  "colors": [
    "blue",
    "green"
  ],
  "materials": [
    "polyester"
  ]
}
